In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import polars as pl # TODO: Convert the whole methodology to either pandas or polars
import tiktoken

import os
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from tqdm import tqdm
from tqdm.notebook import tqdm
from itertools import product

from dotenv import load_dotenv
import os

load_dotenv()

MC_EMAIL = os.environ["MC_EMAIL"]
PUBTATOR_PATH = os.environ["PUBTATOR_PATH"]

# Wags-LLM (v0.2.2) + dgiLIT
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

from dgilit import (
    BioBertEntityTagger,
    CompositeEntityTagger,
    EntityPreTaggingService,
    InteractionClassificationPrompt,
    InteractionClassificationResult,
    PubTator3ChemicalTagger,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
    PubMedArticle
)



# Cancer-focused Interaction Extraction

Description

## Load & Join Data

Read in abstract text that has been retrieved and filtered from previous files.

In [2]:
def load_articles(filename: str | Path) -> dict[str, "PubMedArticle"]:
    """
    Load a previously saved PubMedArticle dictionary.
    """
    with open(filename, "rb") as f:
        return pickle.load(f)

In [3]:
articles = load_articles("../../../data/2026-07-07-test03-articles.pkl")

In [4]:
rows = []

for pmid, article in articles.items():
    for section in article.abstract_sections:
        rows.append(
            {
                "pmid": pmid,
                "section_type": section.section_id,
                "text": section.text,
            }
        )

nih_df = pd.DataFrame(rows)
nih_df = nih_df.rename(columns={"text": "context"})
nih_df.head()

,pmid,section_type,context
0,33495651,ABS1,The research field of ferroptosis has been enj...
1,33597522,Abs1,Understanding global communications among cell...
2,33597522,Abs2,Single-cell methods record molecule expression...
3,33277608,Abs1,"In recent years, the development of nanopartic..."
4,33277608,Abs2,Advances in nanoparticle design could make sub...


In [5]:
pmid_reference = pd.read_csv('../../../data/2026-07-02-test03-oncology-genes-pubtator-pmids.csv')

In [6]:
nih_df["pmid"] = nih_df["pmid"].astype(str)
pmid_reference["pmid"] = pmid_reference["pmid"].astype(str)

gene_map = (
    pmid_reference.groupby("pmid", as_index=False)
       .agg(
           gene_name=("gene_name", "first"),
           gene_mention=("gene_mention", "first"),
       )
)

nih_df = nih_df.merge(gene_map, on="pmid", how="left")

nih_df.head()

,pmid,section_type,context,gene_name,gene_mention
0,33495651,ABS1,The research field of ferroptosis has been enj...,ALK,ALK
1,33597522,Abs1,Understanding global communications among cell...,KIT,kit
2,33597522,Abs2,Single-cell methods record molecule expression...,KIT,kit
3,33277608,Abs1,"In recent years, the development of nanopartic...",EGFR,EGFR
4,33277608,Abs2,Advances in nanoparticle design could make sub...,EGFR,EGFR


## Pre-Tagging

Run the tagging modules to create candidate drug/gene pairs based off the input text. The output of this step are the tagged blocks from previous experiments.

In [7]:
tagger = CompositeEntityTagger(
    BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,
        )
    ),
    PubTator3ChemicalTagger.from_pubtator_file(PUBTATOR_PATH),
)

pretagger = EntityPreTaggingService(
    tagger=tagger,
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [8]:
df = pl.from_pandas(nih_df)

contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)


Device set to use cpu


Tagging text batches:   0%|          | 0/225 [00:00<?, ?it/s]

Device set to use cpu


Tagging text batches:   0%|          | 0/225 [00:00<?, ?it/s]

Normalizing unique entities:   0%|          | 0/4690 [00:00<?, ?entity/s]

In [44]:
import pickle

with open("../../../data/2026-07-07-test03-tagged_blocks.pkl", "wb") as f:
    pickle.dump(tagged_blocks, f, protocol=pickle.HIGHEST_PROTOCOL)

## Cost-Estimation

This section creates one prompt to act as an approximation for the number of token input/output to expect per sample. 

In [39]:
# EXPERIMENTAL
def normalized_entity_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def raw_entity_name(e):
    if e.text:
        return e.text
    return None


def get_candidates(block):
    """Normalized candidates only."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [normalized_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [normalized_entity_name(e)]
        if name
    })

    pmid = block.pmid

    return drugs, genes, pmid


def get_raw_candidates(block):
    """Raw mentions regardless of normalization."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [raw_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [raw_entity_name(e)]
        if name
    })

    return drugs, genes

In [ ]:
PROMPT_TEMPLATE = """
Role:
You are an expert biomedical curator for DGIdb.

Task:
Evaluate whether the supplied drug and gene are supported by the provided text as participating in a drug-gene interaction.

Definitions:
A drug-gene interaction is any observed, inferred, or experimentally supported effect of a drug, compound, ligand, or therapeutic on a gene, gene product, pathway activity, expression, sensitivity, resistance, or binding relationship.

You must determine:

1. Whether evidence supports an interaction.
2. Whether the interaction is activating, inhibiting, or unclear.
3. The most specific interaction type supported by the text.
4. A short direct quote supporting your decision.

Rules: You must use a strict JSON response. Do not be wordy

{{
  "drug": string,
  "gene": string,
  "interaction": boolean,
  "interaction_type": string,
  "directionality": string,
  "evidence": string
}}

Candidate Drug:
{drug}

Candidate Gene:
{gene}

Context:
{context}
"""

def build_prompt(candidate, drug, gene):
    return PROMPT_TEMPLATE.format(
        drug=drug,
        gene=gene,
        context=candidate["context"],
    )

In [ ]:
candidate = {
    "pmid": "33495651",
    "drugs": ["iron carbonyl", "sucrose"],
    "genes": ["ALK"],
    "context": "The research field of ferroptosis has been enjoying exponential growth over the past few years, since the term was coined in 2012. This unique modality of cell death, driven by iron-dependent phospholipid peroxidation, is regulated by multiple cellular metabolic events, including redox homeostasis, iron handling, mitochondrial activity, and metabolism of amino acids, lipids and sugars, in addition to numerous signaling pathways relevant to disease. Intriguingly, therapy-resistant cancer cells, particularly those of the mesenchymal state and prone to metastasis, are exquisitely vulnerable to ferroptosis. Further, numerous organ injuries and degenerative pathologies are driven by ferroptosis. As such, pharmacological modulation of ferroptosis, via both its induction and inhibition, holds great potential for the treatment of drug-resistant cancers, ischemic organ injuries, and other degenerative diseases linked to overwhelming lipid peroxidation. In this Review, we seek to provide an extensive and critical analysis of the current understanding of the molecular mechanisms and regulatory networks of ferroptosis, the potential physiological functions of ferroptosis in tumor suppression and immune surveillance, and its pathological roles and potential for therapeutics. Importantly, as in all rapidly evolving new research areas, issues and confusions exist due to misconceptions and inappropriate use of experimental tools – this Review also tries to address these issues and to provide practical guidelines. Finally, we discuss important concepts and pressing questions that should be a focus of future ferroptosis research.",
}

enc = tiktoken.get_encoding("cl100k_base")

def estimate_tokens(text):
    return len(enc.encode(text))

prompt = build_prompt(candidate, 'sucrose', 'ALK')
estimate_tokens(prompt)

520

In [40]:
# Look up Gene to insert
gene_lookup = dict(
    zip(
        pmid_reference["pmid"].to_list(),
        pmid_reference["gene_name"].to_list(),
    )
)

rows = []
for block in tqdm(
    tagged_blocks,
    total=len(tagged_blocks),
    leave=False,
):
    normalized_drugs, normalized_genes, pmid = get_candidates(block)
    gene = gene_lookup.get(pmid)
    if gene is not None:
        normalized_genes.append(gene)
    number_of_drugs = len(normalized_drugs)
    rows.append({'drugs':normalized_drugs, 'genes': normalized_genes, 'num_drugs': len(normalized_drugs), 'pmid': pmid})
    
cost_check = pd.DataFrame(rows)
cost_check 
cost_check


  0%|          | 0/3585 [00:00<?, ?it/s]

,drugs,genes,num_drugs,pmid
0,"[iron carbonyl, sucrose]",[ALK],2,33495651
1,[],[KIT],0,33597522
2,[],[KIT],0,33597522
3,[],[EGFR],0,33277608
4,[],[EGFR],0,33277608
...,...,...,...,...
3580,[],[MTOR],0,37815057
3581,[rosmarinic acid],[MTOR],1,35630768
3582,[],[EGFR],0,36109501
3583,[],[KRAS],0,36860361


In [46]:
cost_check['num_drugs'][0:100].sum()

np.int64(45)

In [68]:
# Claude Sonnet pricing (USD per million tokens)
INPUT_COST_PER_MILLION = 3.00
OUTPUT_COST_PER_MILLION = 15.00

# Estimated average tokens per request
avg_input_tokens = 520          # measured with tokenizer
avg_output_tokens = 1000      # realistic JSON response estimate

# num_requests = cost_check['num_drugs'].sum()
num_requests = 5000

total_input_tokens = avg_input_tokens * num_requests
total_output_tokens = avg_output_tokens * num_requests

input_cost = total_input_tokens / 1_000_000 * INPUT_COST_PER_MILLION
output_cost = total_output_tokens / 1_000_000 * OUTPUT_COST_PER_MILLION
total_cost = input_cost + output_cost

print(f"Requests:            {num_requests:,}")
print(f"Input tokens:        {total_input_tokens:,}")
print(f"Output tokens:       {total_output_tokens:,}")
print(f"Estimated input:    ${input_cost:.2f}")
print(f"Estimated output:   ${output_cost:.2f}")
print(f"Estimated total:    ${total_cost:.2f}")

Requests:            5,000
Input tokens:        2,600,000
Output tokens:       5,000,000
Estimated input:    $7.80
Estimated output:   $75.00
Estimated total:    $82.80


## LLM Calls

Run each pair from tagged blocks through the LLM classification task. Notably, this experiment run method is slightly different than previous tests to accomodate the use of multiple seed genes (reflecting that a gene set was used as opposed to a single gene).

In [48]:
def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM interaction extraction task runner

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(InteractionClassificationPrompt(version="v2"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

def classify_interaction(
    task_runner: StructuredTaskRunner,
    prompt: InteractionClassificationPrompt,
    context: str,
    candidate_drug: str,
    candidate_gene: str,
) -> InteractionClassificationResult:
    """Classify whether one candidate drug-gene pair is supported by biomedical text."""

    payload = prompt.build_payload(
        context=context,
        candidate_drug=candidate_drug,
        candidate_gene=candidate_gene,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=InteractionClassificationResult,
        )

        return task_result

    except Exception as e:
        return InteractionClassificationResult(
            drug=candidate_drug,
            gene=candidate_gene,
            interaction=False,
            evidence=None,
            interaction_type=None,
            directionality=None,
            error_message=str(e),
        )

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 1000

def run_experiments(
    tagged_blocks,
    temperatures,
    num_runs,
    prompt_version: str,
    use_raw_drug_candidates: bool = False,
    seed_gene: str | None = 'fix this',
):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = InteractionClassificationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                # Look up Gene to insert
                gene_lookup = dict(
                    zip(
                        pmid_reference["pmid"].to_list(),
                        pmid_reference["gene_name"].to_list(),
                    )
                )

                normalized_drugs, normalized_genes, pmid = get_candidates(block)
                gene = gene_lookup.get(pmid)
                if gene is not None:
                    normalized_genes.append(gene)

                raw_drugs, raw_genes = get_raw_candidates(block)

                if use_raw_drug_candidates:
                    candidate_drugs = sorted(set(normalized_drugs) | set(raw_drugs))
                else:
                    candidate_drugs = normalized_drugs

                candidate_genes = normalized_genes

                # TODO: Modify the pydantic to not require a seed_gene
                if seed_gene:
                    candidate_genes_for_classification = candidate_genes
                else:
                    candidate_genes_for_classification = candidate_genes

                if not candidate_drugs or not candidate_genes_for_classification:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No drug and gene candidates",
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes,
                            "normalized_drugs": normalized_drugs,
                            "normalized_genes": normalized_genes,
                            "raw_drugs": raw_drugs,
                            "raw_genes": raw_genes,
                            "classifications": [],
                        }
                    )
                    continue

                classifications = []

                for candidate_drug, candidate_gene in product(
                    candidate_drugs,
                    candidate_genes_for_classification,
                ):
                    result = classify_interaction(
                        task_runner=task_runner,
                        prompt=prompt,
                        context=block.context,
                        candidate_drug=candidate_drug,
                        candidate_gene=candidate_gene,
                    )

                    classifications.append(result.model_dump())

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes_for_classification,
                        "normalized_drugs": normalized_drugs,
                        "normalized_genes": normalized_genes,
                        "raw_drugs": raw_drugs,
                        "raw_genes": raw_genes,
                        "num_candidate_pairs": len(candidate_drugs)
                        * len(candidate_genes_for_classification),
                        "classifications": classifications,
                    }
                )

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "use_raw_drug_candidates": use_raw_drug_candidates,
                    "seed_gene": seed_gene,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [62]:
stored_runs = run_experiments(
    tagged_blocks=tagged_blocks,
    temperatures=[0],
    num_runs=1,
    prompt_version="v2",
    use_raw_drug_candidates=False,
    seed_gene="fix this",
)

Running temp=0, run=1


T=0, run=1:   0%|          | 0/3585 [00:00<?, ?it/s]

Done temp=0, run=1


In [63]:
stored_runs

[{'run_idx': 0,
  'prompt_version': 'v2',
  'temperature': 0,
  'use_raw_drug_candidates': False,
  'seed_gene': 'fix this',
  'results': [{'pmid': '33495651',
    'block_id': '0',
    'skipped': False,
    'candidate_drugs': ['iron carbonyl', 'sucrose'],
    'candidate_genes': ['ALK'],
    'normalized_drugs': ['iron carbonyl', 'sucrose'],
    'normalized_genes': ['ALK'],
    'raw_drugs': ['amino acids', 'iron', 'sugars'],
    'raw_genes': [],
    'num_candidate_pairs': 2,
    'classifications': [{'drug': 'iron carbonyl',
      'gene': 'ALK',
      'interaction': False,
      'evidence': None,
      'interaction_type': None,
      'directionality': None,
      'error_message': 'No evidence of an interaction between iron carbonyl and ALK in the provided text. The abstract discusses ferroptosis mechanisms generally without mentioning ALK or iron carbonyl specifically.'},
     {'drug': 'sucrose',
      'gene': 'ALK',
      'interaction': False,
      'evidence': None,
      'interaction_t

## Save

Export LLM classification results to an external csv file.

In [64]:
def save_experiment_results(
    stored_runs,
    csv_path="../../../data/2026-07-07-test03-interaction_classifications.csv",
    parquet_path="../../../data/2026-07-07-test03-interaction_classifications.parquet",
):
    rows = []

    for run in stored_runs:
        for result in run["results"]:
            candidate_drugs = result.get("candidate_drugs", [])
            candidate_genes = result.get("candidate_genes", [])
            classifications = result.get("classifications", [])

            normalized_drugs = result.get("normalized_drugs", [])
            normalized_genes = result.get("normalized_genes", [])
            raw_drugs = result.get("raw_drugs", [])
            raw_genes = result.get("raw_genes", [])

            base_row = {
                "run_idx": run.get("run_idx"),
                "prompt_version": run.get("prompt_version"),
                "temperature": run.get("temperature"),
                "used_raw_drug_candidates": run.get(
                    "use_raw_drug_candidates",
                    False,
                ),
                "seed_gene": run.get("seed_gene"),

                "pmid": result.get("pmid"),
                "block_id": result.get("block_id"),
                "skipped": result.get("skipped", False),
                "skip_reason": result.get("skip_reason"),

                "candidate_drugs": ";".join(candidate_drugs),
                "candidate_genes": ";".join(candidate_genes),
                "candidate_drug_count": len(candidate_drugs),
                "candidate_gene_count": len(candidate_genes),
                "num_candidate_pairs": result.get("num_candidate_pairs", 0),

                "normalized_drugs": ";".join(normalized_drugs),
                "normalized_genes": ";".join(normalized_genes),
                "normalized_drug_count": len(normalized_drugs),
                "normalized_gene_count": len(normalized_genes),

                "raw_drugs": ";".join(raw_drugs),
                "raw_genes": ";".join(raw_genes),
                "raw_drug_count": len(raw_drugs),
                "raw_gene_count": len(raw_genes),

                "error_message": result.get("error_message"),
            }

            if not classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": None,
                        "gene": None,
                        "interaction": None,
                        "interaction_type": None,
                        "directionality": None,
                        "evidence": None,
                    }
                )
                continue

            for classification in classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": classification.get("drug"),
                        "gene": classification.get("gene"),
                        "interaction": classification.get("interaction"),
                        "interaction_type": classification.get("interaction_type"),
                        "directionality": classification.get("directionality"),
                        "evidence": classification.get("evidence"),
                        "error_message": classification.get("error_message")
                        or result.get("error_message"),
                    }
                )

    df_results = pl.DataFrame(rows)

    df_results.write_csv(csv_path)
    df_results.write_parquet(parquet_path)

    print(df_results.shape)
    return df_results

df_results = save_experiment_results(stored_runs)

df_results.head()

(8320, 29)


run_idx,prompt_version,temperature,used_raw_drug_candidates,seed_gene,pmid,block_id,skipped,skip_reason,candidate_drugs,candidate_genes,candidate_drug_count,candidate_gene_count,num_candidate_pairs,normalized_drugs,normalized_genes,normalized_drug_count,normalized_gene_count,raw_drugs,raw_genes,raw_drug_count,raw_gene_count,error_message,drug,gene,interaction,interaction_type,directionality,evidence
i64,str,i64,bool,str,str,str,bool,str,str,str,i64,i64,i64,str,str,i64,i64,str,str,i64,i64,str,str,str,bool,str,str,str
0,"""v2""",0,false,"""fix this""","""33495651""","""0""",false,null,"""iron carbonyl;sucrose""","""ALK""",2,1,2,"""iron carbonyl;sucrose""","""ALK""",2,1,"""amino acids;iron;sugars""","""""",3,0,"""No evidence of an interaction …","""iron carbonyl""","""ALK""",false,null,null,null
0,"""v2""",0,false,"""fix this""","""33495651""","""0""",false,null,"""iron carbonyl;sucrose""","""ALK""",2,1,2,"""iron carbonyl;sucrose""","""ALK""",2,1,"""amino acids;iron;sugars""","""""",3,0,"""No evidence of interaction bet…","""sucrose""","""ALK""",false,null,null,null
0,"""v2""",0,false,"""fix this""","""33597522""","""1""",true,"""No drug and gene candidates""","""""","""KIT""",0,1,0,"""""","""KIT""",0,1,"""""","""""",0,0,null,null,null,null,null,null,null
0,"""v2""",0,false,"""fix this""","""33597522""","""2""",true,"""No drug and gene candidates""","""""","""KIT""",0,1,0,"""""","""KIT""",0,1,"""""","""""",0,0,null,null,null,null,null,null,null
0,"""v2""",0,false,"""fix this""","""33277608""","""3""",true,"""No drug and gene candidates""","""""","""EGFR""",0,1,0,"""""","""EGFR""",0,1,"""""","""""",0,0,null,null,null,null,null,null,null


In [65]:
df_results['interaction'].value_counts()

interaction,count
bool,u32
false,4410
null,2461
true,1449


In [66]:
df_results['pmid'].n_unique()

2142

In [67]:
len(tagged_blocks)

3585